# 手撕-Norm合集

## BatchNorm

要点：

- 训练时要维护running_mean和running_var，语法不能忘

- mean和var求和的dim不要写错

- 更新running状态参数时需要detach（不要梯度）来从全局计算图中删去

- running_var开始时初始化为全1，防止分母为0

- 方差的unbiased和biased（这个可能没那么重要）


In [2]:
from torch import nn
import torch
class MyBatchNorm2d(nn.Module):
    def __init__(self, num_features, momentum=0.1, epsilon=1e-5): # 第一遍写的时候忘了epsilon, 用于防止方差为0
        super().__init__()
        self.num_features = num_features
        self.momentum = momentum
        self.epsilon = epsilon # !!!
        self.gamma = nn.Parameter(torch.ones(num_features))
        self.beta = nn.Parameter(torch.zeros(num_features))
        
        self.register_buffer('running_mean', torch.zeros(num_features)) # 第一遍写的时候这两个语法不会
        # self.register_buffer('running_var', torch.zeros(num_features)) # 第一遍写错了，应该初始化为torch.ones防止分母为零
        self.register_buffer('running_var', torch.ones(num_features))

    def forward(self, x):
        # (bs, C, W, H)
        bs, C, W, H = x.shape
        if self.training:
            # batch_mean = torch.mean(x,dim=1) 第一遍写错了！dim=1是消去这一维度的求平均，我们要只保留这个维度
            # batch_var = torch.var(x,dim=1)
            batch_mean = x.mean(dim=(0,2,3))
            batch_var_unbiased = x.var(dim=(0,2,3), unbiased=True) # 第一次没写unbiased=True
            batch_var_biased = x.var(dim=(0,2,3), unbiased=False) # 第二次没算biased的方差
            self.running_mean = (1-self.momentum)*self.running_mean + self.momentum*batch_mean.detach() # 第一遍写没加detach，要从计算图中分离
            self.running_var = (1-self.momentum)*self.running_var + self.momentum*batch_var_unbiased.detach()

            mean_to_use = batch_mean.view((1,C,1,1))
            var_to_use = batch_var_biased.view((1,C,1,1)) # 这里需要用biased的方差来更新（实际上应该不需要分这么细）
        else:
            mean_to_use = self.running_mean.view((1,C,1,1))
            var_to_use = self.running_var.view((1,C,1,1))
        
        x_hat = (x-mean_to_use)/torch.sqrt(var_to_use+self.epsilon)
        reshaped_gamma = self.gamma.view((1,C,1,1))
        reshaped_beta = self.beta.view((1,C,1,1))
        out = reshaped_gamma*x_hat + reshaped_beta

        return out


In [3]:
# --- 验证代码 ---
# 设置随机种子以保证结果可复现
torch.manual_seed(42)

# 创建一个随机输入张量
N, C, H, W = 8, 3, 4, 4
input_tensor = torch.randn(N, C, H, W)

# 实例化我们自己写的 BatchNorm
my_bn = MyBatchNorm2d(num_features=C)

# 实例化 PyTorch 官方的 BatchNorm
pytorch_bn = nn.BatchNorm2d(num_features=C)

# --- 验证训练模式 ---
my_bn.train()
pytorch_bn.train()
my_output_train = my_bn(input_tensor)
pytorch_output_train = pytorch_bn(input_tensor.clone()) # 使用clone防止输入被修改

print("--- 训练模式 ---")
# 检查输出是否接近（由于浮点数精度，不会完全相等）
print("输出是否接近:", torch.allclose(my_output_train, pytorch_output_train, atol=1e-5))
# 检查 running_mean 是否更新且接近
print("running_mean是否接近:", torch.allclose(my_bn.running_mean, pytorch_bn.running_mean, atol=1e-5))


# --- 验证推理模式 ---
my_bn.eval()
pytorch_bn.eval()
my_output_eval = my_bn(input_tensor)
pytorch_output_eval = pytorch_bn(input_tensor.clone())

print("\n--- 推理模式 ---")
print("输出是否接近:", torch.allclose(my_output_eval, pytorch_output_eval, atol=1e-5))

--- 训练模式 ---
输出是否接近: True
running_mean是否接近: True

--- 推理模式 ---
输出是否接近: True


## LayerNorm

- 不需要维护running_mean和running_var, 训练和评估时使用完全相同的链路

In [4]:
class MyLayerNorm(nn.Module):
    def __init__(self, normalized_shape, epsilon=1e-5):
        super().__init__()
        # 超参数
        # self.momentum = momentum 不需要!!!
        if isinstance(normalized_shape, int):
            normalized_shape = (normalized_shape,)
        self.normalized_shape = normalized_shape
        self.epsilon = epsilon

        # 可学习的参数
        self.gamma = nn.Parameter(torch.ones(normalized_shape))
        self.beta = nn.Parameter(torch.zeros(normalized_shape))
        # 不需要维护额外的状态!!!
    
    def forward(self, x):
        normalized_dims = tuple(range(-len(self.normalized_shape),0))
        batch_mean = x.mean(dim=normalized_dims, keepdim=True) # 第一次没有keepdim!
        batch_var = x.var(dim=normalized_dims,keepdim=True, unbiased=False) # unbiased=False!!!

        x_hat = (x-batch_mean)/torch.sqrt(batch_var+self.epsilon)
        out = x_hat*self.gamma+self.beta
        return out


In [5]:
# --- 验证脚本 ---
torch.manual_seed(42)

# 定义输入张量的形状
B, L, F = 8, 10, 20 # batch_size, seq_len, num_features
input_tensor = torch.randn(B, L, F)

# 实例化我们自己写的 LayerNorm
# 对于 [B, L, F] 的输入，我们归一化的是最后一个维度 F
my_ln = MyLayerNorm(normalized_shape=F)

# 实例化 PyTorch 官方的 LayerNorm
# 官方的 LayerNorm 接收一个 normalized_shape 参数
pytorch_ln = nn.LayerNorm(normalized_shape=F)

# --- 验证 ---
# 注意：LayerNorm 没有 train 和 eval 模式的区别，所以我们直接测试即可
my_output = my_ln(input_tensor)
pytorch_output = pytorch_ln(input_tensor.clone())

print("--- LayerNorm 验证 ---")
# 检查输出是否接近（由于浮点数精度，不会完全相等）
print("输出是否接近:", torch.allclose(my_output, pytorch_output, atol=1e-5))

# 我们也可以测试更复杂的 normalized_shape
print("\n--- 复杂 shape 验证 ---")
my_ln_complex = MyLayerNorm(normalized_shape=(L, F))
pytorch_ln_complex = nn.LayerNorm(normalized_shape=(L, F))
my_output_complex = my_ln_complex(input_tensor)
pytorch_output_complex = pytorch_ln_complex(input_tensor.clone())
print("复杂 shape 输出是否接近:", torch.allclose(my_output_complex, pytorch_output_complex, atol=1e-5))

--- LayerNorm 验证 ---
输出是否接近: True

--- 复杂 shape 验证 ---
复杂 shape 输出是否接近: True


## RMSNorm

- 是均方根不是方差！！！

- 需要keepdim！！！

In [16]:
class MyRMSNorm(nn.Module):
    def __init__(self, normalized_shape, epsilon=1e-5):
        super().__init__()
        if isinstance(normalized_shape, int):
            normalized_shape = (normalized_shape,)
        self.normalized_shape = normalized_shape
        self.epsilon = epsilon

        self.gamma = nn.Parameter(torch.ones(normalized_shape))

    def forward(self, x):
        normalized_dim = tuple(range(-len(self.normalized_shape),0))
        x_squared = x**2
        batch_rms = x_squared.mean(dim=normalized_dim,keepdim=True)

        x_hat = x/torch.sqrt(batch_rms+self.epsilon)
        out = x_hat*self.gamma
        return out

In [17]:
# --- 验证脚本 ---
torch.manual_seed(42)

# 定义输入张量的形状
B, L, F = 8, 10, 20 # batch_size, seq_len, num_features
input_tensor = torch.randn(B, L, F)

# 实例化我们自己写的 RMSNorm
# 对于 [B, L, F] 的输入，我们归一化的是最后一个维度 F
my_ln = MyRMSNorm(normalized_shape=F)

# 实例化 PyTorch 官方的 RMSNorm
# 官方的 RMSNorm 接收一个 normalized_shape 参数
pytorch_ln = nn.RMSNorm(normalized_shape=F)

# --- 验证 ---
# 注意：RMSNorm 没有 train 和 eval 模式的区别，所以我们直接测试即可
my_output = my_ln(input_tensor)
pytorch_output = pytorch_ln(input_tensor.clone())

print("--- RMSNorm 验证 ---")
# 检查输出是否接近（由于浮点数精度，不会完全相等）
print("输出是否接近:", torch.allclose(my_output, pytorch_output, atol=1e-5))

# 我们也可以测试更复杂的 normalized_shape
print("\n--- 复杂 shape 验证 ---")
my_ln_complex = MyLayerNorm(normalized_shape=(L, F))
pytorch_ln_complex = nn.LayerNorm(normalized_shape=(L, F))
my_output_complex = my_ln_complex(input_tensor)
pytorch_output_complex = pytorch_ln_complex(input_tensor.clone())
print("复杂 shape 输出是否接近:", torch.allclose(my_output_complex, pytorch_output_complex, atol=1e-5))

--- RMSNorm 验证 ---
输出是否接近: True

--- 复杂 shape 验证 ---
复杂 shape 输出是否接近: True
